In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url



In [ ]:
# Connect to database
db = DB.from_url(db_url())
print("Database connected successfully!")


In [ ]:
# Load patients data
print("Loading patients data...")
patients = db.table_df("patients", schema="mimiciii")
print(f"Loaded{len(patients)} patients")
print(f"Shape: {patients.shape}")
print(f"Columns: {list(patients.columns)}")



In [ ]:
# First look at the data
print("First 5 rows:")
display(patients.head())

print("\n Data types:")
print(patients.dtypes)

print("\n Basic statistics:")
display(patients.describe(include='all'))


In [ ]:
# Gender Distribution
print("Gender Distribution:")
gender_counts = patients['gender'].value_counts()
print(gender_counts)

# Visualize gender distribution
plt.figure(figsize=(8, 6))
gender_counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Gender Distribution in MIMIC-III Patients', fontsize=14, fontweight='bold')
plt.xlabel('Gender', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

# Add percentage labels
total = len(patients)
for i, v in enumerate(gender_counts.values):
    plt.text(i, v + total*0.01, f'{v}\n({v/total*100:.1f}%)', 
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/gender_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Query adapted for MIMIC-III database structure
query = """
SELECT
  p.gender,
  EXTRACT(YEAR FROM AGE(a.admittime, p.dob))::int AS age_at_admit
FROM mimiciii.admissions a
JOIN mimiciii.patients p USING (subject_id)
WHERE a.admittime IS NOT NULL AND p.dob IS NOT NULL
"""

# Use your database connection
df = db.query_df(query)

print(f"Loaded {len(df)} admissions")

# keep adults, exclude de-identified ages
df = df[(df["age_at_admit"] >= 18) & (df["age_at_admit"] < 90)]

print(f"Adults (18-89 years): {len(df)} admissions")

# mean ages for annotation
means = df.groupby("gender")["age_at_admit"].mean().round(2)
mean_F = float(means.get("F", float("nan")))
mean_M = float(means.get("M", float("nan")))

print(f"Mean ages - Female: {mean_F:.2f}, Male: {mean_M:.2f}")

plt.figure(figsize=(9, 5))
sns.histplot(
    data=df,
    x="age_at_admit",
    hue="gender",
    bins=range(18, 91),     # 1-year bins
    multiple="stack",       # stacked bars like the example
    edgecolor=None
)
plt.title("Age at admission, by gender")
plt.xlabel("age")
plt.ylabel("count")

# annotate means on the left
ymin, ymax = plt.ylim()
xmin, xmax = plt.xlim()
x_annot = xmin + 0.5
if pd.notna(mean_F):
    plt.text(x_annot, ymax * 0.58, f"F {mean_F:.2f}", ha="left", va="center")
if pd.notna(mean_M):
    plt.text(x_annot, ymax * 0.50, f"M {mean_M:.2f}", ha="left", va="center")

plt.tight_layout()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/age_by_gender.png', dpi=300, bbox_inches='tight')
plt.show()

# Additional statistics
print(f"Total admissions: {len(df):,}")
print(f"Female admissions: {len(df[df['gender'] == 'F']):,}")
print(f"Male admissions: {len(df[df['gender'] == 'M']):,}")

# Age distribution by gender
print(f"\nAge distribution by gender:")
age_stats = df.groupby('gender')['age_at_admit'].agg(['count', 'mean', 'median', 'std']).round(2)
print(age_stats)


In [ ]:
# Age at hospital admission, by admission_type (ALL types, stacked) — fixed colors + full legend

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

query = """
SELECT
  a.admission_type,
  EXTRACT(YEAR FROM AGE(a.admittime, p.dob))::int AS age_at_admit
FROM mimiciii.admissions a
JOIN mimiciii.patients  p USING (subject_id)
WHERE a.admittime IS NOT NULL AND p.dob IS NOT NULL
"""
df = db.query_df(query)

# normalize labels
df["admission_type"] = (
    df["admission_type"].str.upper().str.strip().replace({"EW EMER.": "EMERGENCY"})
)

# adults only; drop de-identified 90+; optionally exclude NEWBORN
df = df[(df["age_at_admit"] >= 18) & (df["age_at_admit"] < 90)]
df = df[df["admission_type"] != "NEWBORN"]   # remove this line if you want every type

# order by frequency
type_order = df["admission_type"].value_counts().index.tolist()
df["admission_type"] = pd.Categorical(df["admission_type"], categories=type_order, ordered=True)

# means for annotation
means = df.groupby("admission_type")["age_at_admit"].mean().round(2)

# choose a stable discrete palette with enough colors
palette = sns.color_palette("tab20", n_colors=len(type_order))
color_map = dict(zip(type_order, palette))

plt.figure(figsize=(12, 5.5))
ax = sns.histplot(
    data=df,
    x="age_at_admit",
    hue="admission_type",
    hue_order=type_order,
    bins=range(18, 91),
    multiple="stack",
    edgecolor=None,
    palette=color_map,
    legend=False,   # we'll build our own legend
)

plt.title("Age at admission, by admission type")
plt.xlabel("age")
plt.ylabel("count")

# manual legend with counts
counts = df["admission_type"].value_counts()
handles = [Patch(facecolor=color_map[t], label=f"{t} (n={counts[t]})") for t in type_order]
ax.legend(
    handles=handles,
    title="admission_type",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0.0,
    frameon=False,
)

# annotate mean ages on the left
ymin, ymax = plt.ylim()
xmin, xmax = plt.xlim()
x_annot = xmin + 0.5
ypos = ymax * 0.80
step = ymax * 0.055
for t in type_order:
    plt.text(x_annot, ypos, f"{t} {means.get(t, float('nan')):.2f}", ha="left", va="center", fontsize=10)
    ypos -= step
    if ypos < ymax * 0.05:
        break  # remove to print all labels even if they overflow

plt.tight_layout()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/age_by_admission_type.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Age at admission, by ethnicity (stacked)

import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Query age at hospital admission + raw ethnicity from MIMIC-III
query = """
SELECT
  a.ethnicity,
  EXTRACT(YEAR FROM AGE(a.admittime, p.dob))::int AS age_at_admit
FROM mimiciii.admissions a
JOIN mimiciii.patients  p USING (subject_id)
WHERE a.admittime IS NOT NULL AND p.dob IS NOT NULL
"""
df = db.query_df(query)   # or: pd.read_sql(query, engine)

# 2) Normalize/Collapse ethnicity labels -> broad buckets
def norm_eth(x: str) -> str:
    if not isinstance(x, str):
        return "UNKNOWN"
    s = x.upper().strip()

    # Common groupings in MIMIC-III
    if "WHITE" in s:
        return "WHITE"
    if "BLACK" in s or "AFRICAN" in s:
        return "BLACK"
    if "HISPANIC" in s or "LATINO" in s:
        return "HISPANIC/LATINO"
    if "ASIAN" in s:
        return "ASIAN"
    if "MIDDLE EASTERN" in s or "PORTUGUESE" in s:
        return "OTHER"
    if "NATIVE" in s or "AMERICAN INDIAN" in s or "ALASKA" in s:
        return "NATIVE/ALASKA"
    if "PACIFIC ISLANDER" in s or "HAWAIIAN" in s:
        return "PACIFIC ISLANDER"
    if "OTHER" in s:
        return "OTHER"
    if "UNKNOWN" in s or "NOT SPECIFIED" in s or "UNABLE TO OBTAIN" in s or "PATIENT DECLINED" in s:
        return "UNKNOWN"
    return "OTHER"

df["ethnicity_group"] = df["ethnicity"].map(norm_eth)

# 3) Adults only; exclude de-identified 90+
df = df[(df["age_at_admit"] >= 18) & (df["age_at_admit"] < 90)]

# 4) Order legend by frequency (most common first)
order = df["ethnicity_group"].value_counts().index.tolist()
df["ethnicity_group"] = pd.Categorical(df["ethnicity_group"], categories=order, ordered=True)

# 5) Means for annotation
means = df.groupby("ethnicity_group")["age_at_admit"].mean().round(2)

# 6) Plot
plt.figure(figsize=(11, 5.5))
sns.histplot(
    data=df,
    x="age_at_admit",
    hue="ethnicity_group",
    hue_order=order,
    bins=range(18, 91),    # 1-year bins
    multiple="stack",
    edgecolor=None,
)
plt.title("Age at admission, by ethnicity")
plt.xlabel("age")
plt.ylabel("count")
plt.legend(title="ethnicity", loc="upper right")

# 7) Annotate mean ages on the left
ymin, ymax = plt.ylim()
xmin, xmax = plt.xlim()
x_annot = xmin + 0.5
ypos = ymax * 0.80
step = ymax * 0.055
for grp in order:
    plt.text(x_annot, ypos, f"{grp} {means.get(grp, float('nan')):.2f}", ha="left", va="center", fontsize=10)
    ypos -= step
    if ypos < ymax * 0.05:
        break  # remove if you want to print *all* groups even if they overflow

plt.tight_layout()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/age_by_ethnicity.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df = db.query_df("SELECT UPPER(TRIM(admission_type)) AS admission_type FROM mimiciii.admissions;")
df["admission_type"] = df["admission_type"].replace({"EW EMER.":"EMERGENCY"})
ax = sns.countplot(data=df, y="admission_type", order=df["admission_type"].value_counts().index)
ax.set_title("Admission type counts"); ax.set_xlabel("count"); ax.set_ylabel("admission_type")
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/admission_type_counts.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df = db.query_df("""
SELECT i.first_careunit,
       EXTRACT(EPOCH FROM (a.dischtime - a.admittime))/86400.0 AS los_days
FROM mimiciii.admissions a
JOIN mimiciii.icustays   i USING (hadm_id)
WHERE a.admittime IS NOT NULL AND a.dischtime IS NOT NULL;
""")
df = df[(df.los_days>=0) & (df.los_days<60)]
sns.histplot(data=df, x="los_days", hue="first_careunit", multiple="stack", bins=60)
plt.title("Days by ICU careunit"); plt.xlabel("days"); plt.ylabel("count")
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/los_by_icu_careunit.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
df = db.query_df("""
SELECT d.long_title, COUNT(*) AS n
FROM mimiciii.diagnoses_icd di
JOIN mimiciii.d_icd_diagnoses d ON di.icd9_code=d.icd9_code
GROUP BY d.long_title
ORDER BY n DESC
LIMIT 15;
""")
sns.barplot(data=df, y="long_title", x="n")
plt.title("Top ICD9 diagnoses"); plt.xlabel("count"); plt.ylabel("diagnosis"); plt.show()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/top_icd9_diagnoses.png', dpi=300, bbox_inches='tight')


In [ ]:
# Top labs and lactate distribution by hospital mortality

import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

sql_top = """
SELECT UPPER(TRIM(d.label)) AS lab, COUNT(*) AS n
FROM mimiciii.labevents l
JOIN mimiciii.d_labitems d ON l.itemid=d.itemid
WHERE l.valuenum IS NOT NULL
GROUP BY lab
ORDER BY n DESC
LIMIT 20;
"""
top = db.query_df(sql_top)

plt.figure(figsize=(9,5))
sns.barplot(data=top, y="lab", x="n")
plt.title("Top lab tests by count"); plt.xlabel("count"); plt.ylabel("lab")
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/top_lab_tests.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Note counts by category + token length histogram
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

sql = """
SELECT category, char_length(text) AS nchar
FROM mimiciii.noteevents
WHERE text IS NOT NULL
"""
notes = db.query_df(sql)
notes["category"] = notes["category"].fillna("UNKNOWN")

plt.figure(figsize=(9,4))
top_cats = notes["category"].value_counts().head(10).index.tolist()
sns.countplot(data=notes[notes["category"].isin(top_cats)],
              y="category", order=top_cats)
plt.title("Top note categories"); plt.xlabel("count"); plt.ylabel("category")
plt.tight_layout()
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/top_note_categories.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7,4))
sns.histplot(notes["nchar"], bins=50)
plt.title("Note length (characters)"); plt.xlabel("chars"); plt.ylabel("count")
plt.savefig('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/note_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
